# Convolutions

Let's look at MNIST. Let's start with a dense-layer model, then build up a conv net layer by layer, if possible.


Todo
- Make a dense model
- Count its params
- Make a conv net
- Compare

In [42]:
import numpy as np
import torch
import os
os.environ['KERAS_BACKEND'] = 'torch'
import keras
from IPython.display import display, Markdown, clear_output
import time

class CustomCallback(keras.callbacks.Callback):
    def __init__(self, epochs):
        super().__init__()
        self.epochs = epochs
    def on_epoch_begin(self, epoch, logs=None):
        c = ['|','/','-','\\']
        print(f"\r{c[epoch % 4]} epoch: {epoch +1}/{self.epochs}", end='')
    def on_train_end(self, logs=None):
        print()

## Dense baseline

In [43]:
# Load MNIST
(train_images, train_labels), (test_images, test_labels) = keras.datasets.mnist.load_data()

x_train_dense = train_images.reshape(60000, 28 * 28).astype("float32") / 255.0
x_test_dense  = test_images.reshape(10000, 28 * 28).astype("float32") / 255.0

test_images = test_images.reshape(10000, 28**2)
test_images = test_images.astype('float32') / 255
y_train_dense = keras.utils.to_categorical(train_labels)
y_test_dense = keras.utils.to_categorical(test_labels)

In [44]:
# model builder
def build_model(input, lr, units, layers):
    keras.backend.clear_session()
    model = keras.models.Sequential()
    model.add(keras.Input((input,)))
    for _ in range(layers):
        model.add(keras.layers.Dense(units, activation='relu'))
    model.add(keras.layers.Dense(10, activation='softmax'))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model


In [45]:
# Baselines

model = build_model(28**2, 0.001, 12, 1)

# untrained model
test_loss, test_acc = model.evaluate(x=x_test_dense, y=y_test_dense)
print(f"baseline untrained loss: {test_loss}, acc: {test_acc}")


# random guess
correct = (np.random.randint(0, y_test_dense.shape[1], size=len(test_labels)) == test_labels).sum()
print(f"baseline random: {correct / len(test_labels)}")



313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.1752 - loss: 2.3030
baseline untrained loss: 2.3030049800872803, acc: 0.17520000040531158
baseline random: 0.1009


So, as expected, both are around 10% accuracy. Although baseline untrained is perhaps higher than I would expect.

Let's train and see how it goes

In [48]:
# log display code
COLUMNS = [
    ("#", lambda e, i: i + 1),
    ("bs", "batch_size"),
    ("lr", "lr"),
    ("units", "units"),
    ("layers", "layers"),
    ("epochs", "epochs"),
    ("params", "params"),
    ("final acc", lambda e, i: f"{e.get('final_acc', 0.0):.4f}"),
    ("best acc", lambda e, i: f"{e.get('best_acc', 0.0):.4f} (epoch {e.get('best_epoch', -1) + 1})"),
    ("time", lambda e, i: f"{e.get('elapsed', 0.0):.1f}s"),
]

def show_experiment_log(log):
    best_i = int(np.argmax([e.get("best_acc", float("-inf")) for e in log]))

    headers = [name for name, _ in COLUMNS]
    header = "| " + " | ".join(headers) + " |\n|" + "|".join(["---"] * len(headers)) + "|"

    rows = []
    for i, e in enumerate(log):
        vals = []
        for name, source in COLUMNS:
            v = source(e, i) if callable(source) else e.get(source, "")
            v = str(v)
            if name == "best acc":
                v = f"**{v}**" if i == best_i else v
            vals.append(v)
        rows.append("| " + " | ".join(vals) + " |")

    clear_output(wait=True)
    display(Markdown(header + "\n" + "\n".join(rows)))


def run_experiment(log, epochs, lr, batch_size, units, layers, K=2):
    num_val_samples = len(x_train_dense) // K # floor div
    all_accuracies = []

    print(f"x train total: {len(x_train_dense)}")
    print(f"samples per fold {num_val_samples}")

    start = time.time()

    for i in range(K):
        print('processing fold', i)
        a = i * num_val_samples
        b = a + num_val_samples

        # Validation data
        val_data = x_train_dense[a:b]
        val_targets = y_train_dense[a:b]

        # Training data
        partial_x_train = np.concatenate((x_train_dense[:a], x_train_dense[b:]))
        partial_train_targets = np.concatenate((y_train_dense[:a], y_train_dense[b:]))

        model = build_model(28**2, lr, units, layers)

        history = model.fit(partial_x_train, 
                partial_train_targets,
                batch_size=batch_size,
                epochs=epochs,
                validation_data=(val_data, val_targets),
                verbose=0,
                callbacks=[CustomCallback(epochs)],
                )

        all_accuracies.append(history.history['val_accuracy'])


    average_accuracy = np.array(all_accuracies).mean(axis=0)

    elapsed = time.time() - start

    log.append(dict(
        epochs=epochs, lr=lr, batch_size=batch_size,
        units=units, layers=layers, params=model.count_params(),
        final_acc=average_accuracy[-1],
        best_epoch=int(np.argmax(average_accuracy)),
        best_acc=average_accuracy[int(np.argmax(average_accuracy))],
        elapsed=elapsed,
    ))
    show_experiment_log(log)

In [49]:
log=[]
run_experiment(log, 1, 0.002, 128, 12, 1)
run_experiment(log, 1, 0.002, 128, 32, 2)
run_experiment(log, 1, 0.002, 128, 32, 1)

| # | bs | lr | units | layers | epochs | params | final acc | best acc | time |
|---|---|---|---|---|---|---|---|---|---|
| 1 | 128 | 0.002 | 12 | 1 | 1 | 9550 | 0.8907 | 0.8907 (epoch 1) | 7.0s |
| 2 | 128 | 0.002 | 32 | 2 | 1 | 26506 | 0.9136 | **0.9136 (epoch 1)** | 7.7s |
| 3 | 128 | 0.002 | 32 | 1 | 1 | 25450 | 0.9130 | 0.9130 (epoch 1) | 7.2s |

So, we get around 94% accuracy with 26,506 params.

# CNN Architecture

### Conv2D layers


In a Conv2D layer, a unit is called a "filter" or "output channel".

#### Most basic

Say we have:
- 1 channel
- a single Conv2D layer with a single filter
- the input and output of the filter is the same rez (10x10)
- A kernel size of 3

Then, we can think of the filter unit as a CUDA kernel that runs on each possible 3x3 neighborhood (so it's a sliding window).

The filter unit has 9 trainable weights (one per kernel cell) and one trainable bias.



#### Input and outputs different

Say:
- 1 channel
- Single conv2Dlayer with filters=1 (a single filter)
- input is 10x10, output is 5x2
- kernel size 3

We still have 9 weights (one per cell) and one bias for the filter unit.

But now we have a stride of 2. So instead of each kernel having a 1px offset, each kernel has a 2px offset. 



#### Multiple filters in a layer

Say:
- 1 channel
- Single conv2Dlayer with filters=8 
- input is 10x10, output is 5x2
- kernel size 3


Now we have 8 filter units, so after this layer we are now working in 8 channel space. That is, each filter unit, since it runs on every single kernel, produces a new texture. So we end up with 8 textures if we have 8 filter units.

With a dense layer, each unit has a weight for each unit in the previous layer. In a conv2d layer, the architecture is very different. 

Here, each unit has 9 weights and a bias. So the number of parameters is determined by the kernel size, rather than the input size.


### Max pooling
This is a downsampling layer. Has no trainable params.

Takes a window (eg 2x2) and then outputs the max value in that set. 

In [51]:
# For a CNN, we don't flatten the input data, since spatial locality is used
(train_images, y_train), (test_images, y_test) = keras.datasets.mnist.load_data()
x_train = train_images.reshape((60000, 28, 28, 1)).astype('float32') / 255
x_test = test_images.reshape((10000, 28, 28, 1)).astype('float32') / 255

In [ ]:

# log display code
COLUMNS = [
    ("#", lambda e, i: i + 1),
    ("bs", "batch_size"),
    ("lr", "lr"),
    ("filter_layers", "filter_layers"),
    ("kernel_size", "kernel_size"),
    ("epochs", "epochs"),
    ("params", "params"),
    ("final acc", lambda e, i: f"{e.get('final_acc', 0.0):.4f}"),
    ("best acc", lambda e, i: f"{e.get('best_acc', 0.0):.4f} (epoch {e.get('best_epoch', -1) + 1})"),
    ("time", lambda e, i: f"{e.get('elapsed', 0.0):.1f}s"),
]

def show_experiment_log(log):
    best_i = int(np.argmax([e.get("best_acc", float("-inf")) for e in log]))

    headers = [name for name, _ in COLUMNS]
    header = "| " + " | ".join(headers) + " |\n|" + "|".join(["---"] * len(headers)) + "|"

    rows = []
    for i, e in enumerate(log):
        vals = []
        for name, source in COLUMNS:
            v = source(e, i) if callable(source) else e.get(source, "")
            v = str(v)
            if name == "best acc":
                v = f"**{v}**" if i == best_i else v
            vals.append(v)
        rows.append("| " + " | ".join(vals) + " |")

    clear_output(wait=True)
    display(Markdown(header + "\n" + "\n".join(rows)))

# model builder
def build_model(lr,filter_layers, kernel_size ):
    keras.backend.clear_session()
    x = inputs = keras.Input(shape=(28,28,1))
    for filters in filter_layers:
        x = keras.layers.Conv2D(filters=filters, kernel_size=kernel_size, activation='relu')(x)
        x = keras.layers.MaxPooling2D(pool_size=2)(x)

    x = keras.layers.GlobalAveragePooling2D()(x)
    outputs=keras.layers.Dense(10, activation='softmax')(x)
    model = keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=['accuracy']
    )
    return model

def run_experiment(log,epochs, lr, batch_size, filter_layers, kernel_size, K=1):
    all_accuracies = []

    print(f"x train total: {len(x_train)}")

    start = time.time()
    num_val_samples = len(x_train) // K # floor div


    for i in range(K):
            
        a = i * num_val_samples
        b = a + num_val_samples

        if(K==1):
            a = 0
            b = round(len(x_train) * 0.1)
        else:
            print('processing fold', i)
        # Validation data
        val_data = x_train[a:b]
        val_targets = y_train[a:b]

        # Training data
        partial_x_train = np.concatenate((x_train[:a], x_train[b:]))
        partial_train_targets = np.concatenate((y_train[:a], y_train[b:]))

        model = build_model(lr,filter_layers, kernel_size)

        history = model.fit(partial_x_train, 
                partial_train_targets,
                batch_size=batch_size,
                epochs=epochs,
                validation_data=(val_data, val_targets),
                verbose=0,
                callbacks=[CustomCallback(epochs)],
                )

        all_accuracies.append(history.history['val_accuracy'])


    average_accuracy = np.array(all_accuracies).mean(axis=0)

    elapsed = time.time() - start

    log.append(dict(
        epochs=epochs, lr=lr, batch_size=batch_size,
        filter_layers=filter_layers, kernel_size=kernel_size, params=model.count_params(),
        final_acc=average_accuracy[-1],
        best_epoch=int(np.argmax(average_accuracy)),
        best_acc=average_accuracy[int(np.argmax(average_accuracy))],
        elapsed=elapsed,
    ))
    show_experiment_log(log)
    

In [53]:
log=[]
run_experiment(log, 5, 0.001, 128, [64, 128], 2)

| # | bs | lr | filter_layers | kernel_size | epochs | params | final acc | best acc | time |
|---|---|---|---|---|---|---|---|---|---|
| 1 | 128 | 0.001 | [64, 128] | 2 | 5 | 34506 | 0.7845 | **0.7845 (epoch 5)** | 28.1s |

In this initial experiment, the CNN is much worse (78%) than the dense net (90%) even though it has more epochs.

In [54]:
# try larger kernel size
run_experiment(log, 5, 0.001, 128, [64, 128], 3)

| # | bs | lr | filter_layers | kernel_size | epochs | params | final acc | best acc | time |
|---|---|---|---|---|---|---|---|---|---|
| 1 | 128 | 0.001 | [64, 128] | 2 | 5 | 34506 | 0.7845 | 0.7845 (epoch 5) | 28.1s |
| 2 | 128 | 0.001 | [64, 128] | 3 | 5 | 75786 | 0.9435 | **0.9435 (epoch 5)** | 28.5s |

With kernel size 3, we get to 94%. Training is much slower than dense though.

In [55]:
run_experiment(log, 5, 0.001, 128, [32, 64], 3)

| # | bs | lr | filter_layers | kernel_size | epochs | params | final acc | best acc | time |
|---|---|---|---|---|---|---|---|---|---|
| 1 | 128 | 0.001 | [64, 128] | 2 | 5 | 34506 | 0.7845 | 0.7845 (epoch 5) | 28.1s |
| 2 | 128 | 0.001 | [64, 128] | 3 | 5 | 75786 | 0.9435 | **0.9435 (epoch 5)** | 28.5s |
| 3 | 128 | 0.001 | [32, 64] | 3 | 5 | 19466 | 0.9167 | 0.9167 (epoch 5) | 31.4s |

In [57]:

log=[]
run_experiment(log, 2, 0.002, 64, [32, 64], 3)
run_experiment(log, 2, 0.001, 64, [32, 64], 3)
run_experiment(log, 2, 0.002, 128, [32, 64], 3)
run_experiment(log, 2, 0.002, 128, [32, 64], 6)
run_experiment(log, 2, 0.002, 128, [32, 64], 4)
run_experiment(log, 2, 0.002, 128, [32, 32], 4)

| # | bs | lr | filter_layers | kernel_size | epochs | params | final acc | best acc | time |
|---|---|---|---|---|---|---|---|---|---|
| 1 | 64 | 0.002 | [32, 64] | 3 | 2 | 19466 | 0.9083 | 0.9083 (epoch 2) | 21.5s |
| 2 | 64 | 0.001 | [32, 64] | 3 | 2 | 19466 | 0.8990 | 0.8990 (epoch 2) | 22.0s |
| 3 | 128 | 0.002 | [32, 64] | 3 | 2 | 19466 | 0.8972 | 0.8972 (epoch 2) | 11.5s |
| 4 | 128 | 0.002 | [32, 64] | 6 | 2 | 75626 | 0.9658 | **0.9658 (epoch 2)** | 12.2s |
| 5 | 128 | 0.002 | [32, 64] | 4 | 2 | 34026 | 0.9408 | 0.9408 (epoch 2) | 11.8s |
| 6 | 128 | 0.002 | [32, 32] | 4 | 2 | 17290 | 0.9223 | 0.9223 (epoch 2) | 11.4s |

So, in 1 epoch the CNN gets to the level of the dense network.

Increasing kernel size has a big impact.